# 策略概述

**Agglomerative（yF / FMP）** 是命題 1 的**主力**：用「價格行為 ⊕ 公司基本面（規模、估值）⊕ 產業」的**混合特徵**做階層聚類建立配對搜尋空間。

直覺：基本面相近（規模相當、估值層級相同）的公司，更可能對相同的宏觀與估值衝擊同向反應，其價格間的長期均衡關係有基本面基礎。本策略有兩個資料源變體，方法論完全相同、僅基本面資料來源不同（見下表）。實測為**全研究最強的誠實策略**。

| 變體 | 基本面資料來源 | 前視偏誤 |
| :--- | :--- | :--- |
| **yF** | yfinance **單一時點快照** | 有（每個歷史窗都用「今日」數值） |
| **FMP** | **Point-in-Time 逐點**資料 | 已修正（每窗只取形成期末當時可得的最近一筆） |


# 策略架構

與 HDBSCAN 版一樣**只替換「分組」層**——這裡改用「價格 + 基本面 + 產業」混合特徵的階層聚類；其餘篩選、排序、交易與傳統基準完全相同。

```{mermaid}
flowchart LR
  P["混合特徵<br/>價格因子 ⊕ 基本面 ⊕ 產業"] --> G["分組<br/>Agglomerative 階層聚類"]
  G --> F["篩選<br/>群內共整合 + 半衰期 + Hurst"]
  F --> R["排序<br/>最小 SSD 距離"]
  R --> T["Top N 配對<br/>→ 交易期"]
```

| 層 | 本策略採用 | 用途 |
| :--- | :--- | :--- |
| 特徵 | 價格因子載荷 ⊕ 規模/估值 ⊕ 產業別 | 讓「行為＋基本面」都相近的股票靠近 |
| 分組 | Agglomerative 階層聚類 | 資料驅動分群，取代 GICS |
| 篩選 | 群內共整合 + 半衰期 + Hurst | 確認價差均值回歸 |
| 排序 | 最小 SSD 距離 | 挑走勢最接近的對 |
| 交易 | Z-Score（標準化空間） | 偏離進場、回歸出場 |


## 為何用 Agglomerative 階層聚類

- **每個點都分配群組**（無「雜訊」概念）；不適合成群者以「過小群併入 Unknown」事後排除，排除比例由資料決定
- **不預設群數**：以每期合併距離的分位數校準 `distance_threshold`，群數隨當期分布自然決定
- **average linkage**：對混合特徵（連續 + one-hot）的距離尺度差異較不敏感，降低單一巨型群傾向


## yF 與 FMP 的關鍵差異：前視偏誤

兩變體唯一的實質差異在基本面資料的**時間對齊**：

- **yF**：市值/本益比為 yfinance 抓取當下的**單一快照**，所有 2000–2025 歷史窗口共用同一組「現在的」數值——
  對早期窗口存在前視偏誤（用未來資訊分群），是免費資料源下已知且刻意接受的限制
- **FMP**：從 Point-in-Time Parquet 中，對每個形成窗**取日期 ≤ 形成期末的最近一筆**基本面記錄，
  並優先採用該時點的 PIT 產業分類——每個窗口只用當時可得的資訊，**修正了 yF 的前視偏誤**

FMP 變體是為了在保留基本面分群方法的前提下，消除 yF 的前視限制而建立的對照。


# 參考文獻與引用對應


## 文獻 1：Hong & Hwang (2021)

> Hong, S., & Hwang, S. (2021). In search of pairs using firm fundamentals: Is pairs trading profitable? *The European Journal of Finance*, **29**(5).

**參考部分**：以企業基本面特徵（而非純價格）識別配對；基本面相近的公司共享估值與現金流驅動因子，價格均衡有基本面基礎。

**為何參考**：混合特徵中基本面區塊（log 市值、盈餘殖利率 $1/PE$）的直接依據。



## 文獻 2：Ward (1963)／Avellaneda & Lee (2010)

> Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *JASA*, **58**(301), 236–244.
> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7), 761–782.

**參考部分與理由**：

- Ward：階層聚集聚類框架與 dendrogram 可事後切割的特性——階段 3 兩段式 `distance_threshold` 校準之依據
- Avellaneda & Lee：報酬 PCA 因子載荷——價格行為區塊的計算方式


## 文獻 3：Gatev, Goetzmann & Rouwenhorst (2006)／Engle & Granger (1987)／Krauss et al. (2016)

**參考部分與理由**：群內 min-SSD 排序（Gatev）＋ ADF 共整合（Engle & Granger）＋ 半衰期/Hurst 過濾（Krauss）。


# 各階段行為


## 階段 1：混合特徵矩陣建構

三區塊特徵：

- **價格行為（5 維）**：借用 價格因子載荷萃取步驟 的報酬 PCA 因子載荷（不用其聚類）
- **公司基本面（2 維）**：$[\log(1+\text{MarketCap}),\ 1/PE]$；缺值以正規化產業中位數插補後 winsorize（1%–99%）
- **產業歸屬（12 維）**：11 個正規化 GICS 產業 + Unknown 的 one-hot（產業別名先正規化，如 Healthcare→Health Care）

**資料源差異**：yF 從 SQLite 快照取 MarketCap/TrailingPE；FMP 從 PIT Parquet 取形成期末當時的 market_cap/pe_ratio 與 PIT 產業。


## 階段 2：區塊分別標準化與加權拼接

$$X = [\, w_{price}\cdot\text{Scale}(\text{loadings}) \ \|\ w_{fund}\cdot\text{Scale}(f) \ \|\ w_{sector}\cdot\text{OneHot} \,], \quad w_{price}=w_{fund}=w_{sector}=1.0$$

三區塊**各自** `StandardScaler` 後加權拼接——避免 joint 標準化讓 12 欄 one-hot 稀釋 2 欄基本面的距離貢獻。


## 階段 3：Agglomerative 分群與 distance_threshold 校準（依據：Ward 1963）

1. **Probe**：`n_clusters=1` 跑完整合併路徑（`compute_distances=True`），取全部 $N-1$ 個合併距離
2. **校準**：$\text{threshold} = \text{percentile}_{75}(\{d_{merge}\})$
3. **分群**：average-linkage 階層聚類（依合併距離分位數校準切割門檻）；過小群（< `min_cluster_size`=5）併入 `"Unknown"`


## 階段 4：群內 min-SSD 排序與過濾（依據：Gatev 2006）

分群標籤當 `sector_mapping` 餵給 `ssd_rolling.Formation`：群內 Z-Score 標準化 log-price → SSD 初篩 →
協方差 OLS 對沖比例 → 三道過濾（ADF $p<0.05$、半衰期 $[1,42]$ 日、Hurst $<0.5$）→ 依 SSD 升序取前 `top_n`。


## 階段 5–6：輸出與交易期銜接

輸出標準欄位 + `Sector_A/B`（真實 GICS）、`Cluster_ID_A/B`、`MarketCap_A/B`、`TrailingPE_A/B`。
排序流程不輸出 `OLS_Alpha`，交易期於標準化空間重建 spread（路徑 B）。
形成期統計量整個交易期凍結（無前視——但 yF 變體另有基本面快照的前視限制，見「關鍵差異」一節）。
本策略配對另供 DRL 門檻選擇式交易端使用（見 `trading/drl_threshold_trading.ipynb`）。


# 參數總表

| 參數 | 值 | 對應階段 | 說明 |
| :--- | :---: | :--- | :--- |
| 形成窗 / 滾動步長 | 252 / 21 交易日 | 全流程輸入 | 約一年 / 一個月 |
| 每期配對數 | 網格 [1, 3, 5, 10, 20] | 排序（選取） | 依 SSD 升序取前幾組 |
| 價格因子數 | 5 | 特徵 | 價格行為區塊維度 |
| 三區塊權重 | 各 1.0 | 特徵 | 價格 / 基本面 / 產業 等權 |
| 分群連結法 / 門檻分位 | average / 75% | 分組 | 階層聚類的合併規則與切割門檻 |
| 最小群大小 | 5 | 分組 | 過小群併入「未分類」而排除 |
| 共整合顯著水準 | 0.05 | 篩選 | 群內共整合檢定門檻 |
| 基本面資料源 | yF 快照 / FMP 逐點 | 特徵 | 兩變體差異所在 |
